<a href="https://colab.research.google.com/github/stevenolanecon/7002LBSAI/blob/main/notebooks/week8_inclass_tasks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Before you start:**

- Go to **File → Save a copy in Drive**. Until you do this, your work only exists in this browser tab – you can't save changes back to GitHub, and closing the tab or losing the session will lose your work.
- Turn off Colab's autocomplete: **Tools → Settings → Editor → uncheck "Show context-powered code completions."** It's a setting on your own Google account, not something built into this notebook, so you'll need to do it once per account.

# Week 8 – In-class tasks

Work through these during the workshop, alongside the slides. Apart from one hint in Task 4, no syntax hints here beyond what's in the setup cell – `smf.ols` / `smf.logit` / `smf.probit` all take the same formula-string syntax you already know.

`marginaleffects` is pre-installed in Colab, so `import marginaleffects as me` will just work.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import marginaleffects as me

data_path = "https://raw.githubusercontent.com/stevenolanecon/7002LBSAI/main/data/share_health.csv"
df = pd.read_csv(data_path)
df['country'] = df['country'].astype(str).astype('category')  # text labels, one dummy per country
df.head()

## Task 1 – fit the LPM yourself

**Business question:** how big is the raw gap between smokers and non-smokers, before any controls?

Fit `stayshealthy ~ smoking` with `smf.ols` and `cov_type='HC1'`. What's the coefficient, and how would you phrase it as a sentence a health policy team could use?

Then: does that survive adding controls for age, exercise, income, education (and the rest of the full set: `ever_smoked`, `female`, `eduyears`, `income10`, `bmi`, `exerc`, plus `country` for country fixed effects – the setup cell has already made it a category)?

## Task 2 – fit all three yourself

**Business question:** does the *smoking* coefficient tell the same story whichever model you use?

Using the full model with controls, fit all three: `smf.ols`, `smf.logit`, `smf.probit` – same formula each time. Add `country` to the formula: the setup cell has already made it a category, so each country gets its own dummy rather than being read as one continuous number.

Compare the three `smoking` coefficients in one table.

## Task 3 – compute an AME yourself

**Business question:** does exercising regularly matter as much as not smoking?

Fit the full logit model yourself (country included, as in Task 2) and use `marginaleffects.avg_slopes()` to get the AME for `exerc`. Compare its size to smoking's AME.

## Task 4 – does probit agree with logit?

**Business question:** we've been treating logit and probit as interchangeable. Are they, really?

Fit both `smf.logit` and `smf.probit` on the full model. Pick one hypothetical person – a 55-year-old female smoker, sample-average on everything else – and use `.predict()` on a one-row dataframe to get each model's predicted probability for that person. How close are they?

> **Syntax hint – a one-row dataframe for one person.** `.predict()` needs a value for every variable in the formula, each as a one-item list. Set the ones that describe your person, use the sample average for the rest, and the most common category for a categorical variable:
> ```python
> person = pd.DataFrame({'x1': [1],                    # a dummy, set to 1
>                        'x2': [55],                   # a number you choose
>                        'x3': [df['x3'].mean()],      # sample average
>                        'g':  [df['g'].mode()[0]]})   # most common category
> model.predict(person)
> ```

## Task 5 – compute the Brier score yourself

**Business question:** how would you check a reported goodness-of-fit table wasn't just typed in?

The Brier score is nothing more than the same squared-error idea behind R² back in Week 4 – just applied to a probability instead of a price. Get logit's predicted probabilities with `.predict()`, then compute `np.mean((pred - y) ** 2)` by hand.